In [155]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
import torch 
from paddleocr import PaddleOCR
import paddle
import pprint
import numpy as np
from PIL import Image

In [156]:
gpu_or_cpu="cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(gpu_or_cpu) #used to set the model to run in cpu or cuda 

print("Using device:", device)

model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base")
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base",apply_ocr=False)

# Move model to GPU
model.to(device)

Using device: cuda


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.weight                  | MISSING    | 
classifier.bias                    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LayoutLMv3ForTokenClassification(
  (layoutlmv3): LayoutLMv3Model(
    (embeddings): LayoutLMv3TextEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (x_position_embeddings): Embedding(1024, 128)
      (y_position_embeddings): Embedding(1024, 128)
      (h_position_embeddings): Embedding(1024, 128)
      (w_position_embeddings): Embedding(1024, 128)
    )
    (patch_embed): LayoutLMv3PatchEmbeddings(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
    (encoder): LayoutLMv3Encoder

In [157]:
image_path = "./training data/1.jpg"

In [158]:
print(paddle.device.get_device())
ocr = PaddleOCR(use_textline_orientation=True, lang='en', device='gpu')
result = ocr.predict(image_path)

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


gpu:0


Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


In [159]:
pprint.pprint(result)

[{'doc_preprocessor_res': {'angle': 0,
                           'input_img': array([[[51, ..., 48],
        ...,
        [39, ..., 49]],

       ...,

       [[30, ..., 46],
        ...,
        [23, ..., 14]]], shape=(3300, 2232, 3), dtype=uint8),
                           'input_path': None,
                           'model_settings': {'use_doc_orientation_classify': True,
                                              'use_doc_unwarping': True},
                           'output_img': array([[[255, ..., 255],
        ...,
        [255, ..., 255]],

       ...,

       [[255, ..., 255],
        ...,
        [255, ..., 255]]], shape=(3300, 2232, 3), dtype=uint8),
                           'page_index': None,
                           'rot_img': array([[[51, ..., 48],
        ...,
        [39, ..., 49]],

       ...,

       [[30, ..., 46],
        ...,
        [23, ..., 14]]], shape=(3300, 2232, 3), dtype=uint8)},
  'dt_polys': [array([[82, 15],
       ...,
       [82, 59]], sha

In [160]:
result

[{'input_path': './training data/1.jpg',
  'page_index': None,
  'doc_preprocessor_res': {'input_path': None,
   'page_index': None,
   'input_img': array([[[51, ..., 48],
           ...,
           [39, ..., 49]],
   
          ...,
   
          [[30, ..., 46],
           ...,
           [23, ..., 14]]], shape=(3300, 2232, 3), dtype=uint8),
   'model_settings': {'use_doc_orientation_classify': True,
    'use_doc_unwarping': True},
   'angle': 0,
   'rot_img': array([[[51, ..., 48],
           ...,
           [39, ..., 49]],
   
          ...,
   
          [[30, ..., 46],
           ...,
           [23, ..., 14]]], shape=(3300, 2232, 3), dtype=uint8),
   'output_img': array([[[255, ..., 255],
           ...,
           [255, ..., 255]],
   
          ...,
   
          [[255, ..., 255],
           ...,
           [255, ..., 255]]], shape=(3300, 2232, 3), dtype=uint8)},
  'dt_polys': [array([[82, 15],
          ...,
          [82, 59]], shape=(4, 2), dtype=int16),
   array([[1264,   1

In [161]:
pprint.pprint(len(result[0]["rec_texts"]))
pprint.pprint(result[0]["rec_texts"])

147
['PARMESHWARA STATIONERY',
 'Involce No.',
 'e-Way Bill No.',
 'Dated',
 'Near Hasanamba Temple Circle, Balegara Street',
 'PS/25-26/1712',
 '18-Sep-26',
 'Delivery Note',
 'Mode/Terms of Payment',
 'Hassan - 573201',
 'Mob : 9448919881, 7760638884',
 'GSTIN/UIN: 29AFHPM3915J1Z5',
 'Reference No. & Date.',
 'Other References',
 'State Name : Kamataka, Code : 29',
 'Buyer (Bill to)',
 "Buyer's Order No.",
 'Dated',
 'MCE CES',
 'Malnad College of Engineering',
 'Dispatch Doc No',
 'Delivery Note Date',
 'Salagame Rd, Rangoli Halla',
 'Hassan',
 'Dispatched through',
 'Destination',
 'State Name',
 ': Karnataka, Code : 29',
 'Terms of Delivery',
 'SI',
 'Description of Goods',
 'HSN/SAC',
 'GST',
 'Quantity',
 'Rate',
 'Rate',
 'per',
 'Amount',
 'No.',
 'Rate',
 '..',
 '(Incl. of Tax)',
 '1',
 'Casio FX 991 ES Plus Calculator',
 '84701000',
 '18 %',
 '50 Pcs',
 '1,225.01',
 '1,038.14',
 'Pcs',
 '51,907.00',
 '2',
 'Omega Roller Scale 16cms',
 '90172010',
 '18 %',
 '160 Pcs',
 '38.00

In [162]:
pprint.pprint(len(result[0]["rec_polys"]))
pprint.pprint(result[0]["rec_polys"])

147
[array([[82, 15],
       ...,
       [82, 59]], shape=(4, 2), dtype=int16),
 array([[1264,   11],
       ...,
       [1265,   52]], shape=(4, 2), dtype=int16),
 array([[1529,    7],
       ...,
       [1530,   53]], shape=(4, 2), dtype=int16),
 array([[1751,    6],
       ...,
       [1752,   45]], shape=(4, 2), dtype=int16),
 array([[ 82,  65],
       ...,
       [ 82, 109]], shape=(4, 2), dtype=int16),
 array([[1265,   63],
       ...,
       [1266,  104]], shape=(4, 2), dtype=int16),
 array([[1752,   57],
       ...,
       [1753,  103]], shape=(4, 2), dtype=int16),
 array([[1268,  110],
       ...,
       [1269,  148]], shape=(4, 2), dtype=int16),
 array([[1756,  103],
       ...,
       [1756,  144]], shape=(4, 2), dtype=int16),
 array([[ 86, 114],
       ...,
       [ 86, 148]], shape=(4, 2), dtype=int16),
 array([[ 82, 154],
       ...,
       [ 82, 194]], shape=(4, 2), dtype=int16),
 array([[ 84, 198],
       ...,
       [ 84, 236]], shape=(4, 2), dtype=int16),
 array([[127

In [163]:
pprint.pprint(result[0]["rec_texts"][0])
pprint.pprint(result[0]["rec_polys"][0])

print(result[0]["rec_polys"][0][:])
poly=result[0]["rec_polys"][0][:]

'PARMESHWARA STATIONERY'
array([[82, 15],
       ...,
       [82, 59]], shape=(4, 2), dtype=int16)
[[82 15]
 ...
 [82 59]]


In [164]:

# to get the max and min points 
def getMinMaxPoint(poly):
    x0 = int(poly[:,0].min())
    y0 = int(poly[:,1].min())
    x1 = int(poly[:,0].max())
    y1 = int(poly[:,1].max())
    return [x0, y0, x1, y1] 

print(getMinMaxPoint(poly))
 # the rectangle for the **entire line**


[82, 15, 675, 66]


In [165]:
pprint.pprint(result[0]["rec_texts"][0].split())
a=result[0]["rec_texts"][0].split()

print(len(a[0].strip()))

print(result[0]["rec_texts"][0])
print(len(result[0]["rec_texts"][0]))
print(len(a[0]),len(a[1]))

['PARMESHWARA', 'STATIONERY']
11
PARMESHWARA STATIONERY
22
11 10


In [166]:
token_word=[]
token_poly=[]

In [167]:
xy_points=getMinMaxPoint(poly)
sentence= result[0]["rec_texts"][0]
sentence_split = result[0]["rec_texts"][0].split()

def tokenize_sent(xy_points, sentence_split, sentence):
    letter_width=(xy_points[2]-xy_points[0])/len(sentence)
    x = xy_points[0]
    y0, y1 = xy_points[1], xy_points[3]

    for i, word in enumerate(sentence_split):
        word_width = len(word) * letter_width
        x_end = x + word_width

        # print([int(x), y0, int(x_end), y1])
        token_word.append(word)
        token_poly.append([int(x), y0, int(x_end), y1])
        x = x_end

        # add space after word (except last)
        if i < len(sentence_split) - 1:
            x += letter_width


print(tokenize_sent(xy_points,sentence_split,sentence))

None


In [168]:
token_word
token_poly

[[82, 15, 378, 66], [405, 15, 675, 66]]

In [169]:
for i,(box,word) in enumerate(zip(result[0]["rec_polys"],result[0]["rec_texts"])):
    points_max_min=getMinMaxPoint(box[:])
    tokenize_sent(points_max_min,word.split(),word)
    print(box[:],word)

[[82 15]
 ...
 [82 59]] PARMESHWARA STATIONERY
[[1264   11]
 ...
 [1265   52]] Involce No.
[[1529    7]
 ...
 [1530   53]] e-Way Bill No.
[[1751    6]
 ...
 [1752   45]] Dated
[[ 82  65]
 ...
 [ 82 109]] Near Hasanamba Temple Circle, Balegara Street
[[1265   63]
 ...
 [1266  104]] PS/25-26/1712
[[1752   57]
 ...
 [1753  103]] 18-Sep-26
[[1268  110]
 ...
 [1269  148]] Delivery Note
[[1756  103]
 ...
 [1756  144]] Mode/Terms of Payment
[[ 86 114]
 ...
 [ 86 148]] Hassan - 573201
[[ 82 154]
 ...
 [ 82 194]] Mob : 9448919881, 7760638884
[[ 84 198]
 ...
 [ 84 236]] GSTIN/UIN: 29AFHPM3915J1Z5
[[1273  205]
 ...
 [1273  235]] Reference No. & Date.
[[1758  200]
 ...
 [1758  234]] Other References
[[ 78 239]
 ...
 [ 78 286]] State Name : Kamataka, Code : 29
[[ 80 286]
 ...
 [ 80 332]] Buyer (Bill to)
[[1272  300]
 ...
 [1272  331]] Buyer's Order No.
[[1754  299]
 ...
 [1756  335]] Dated
[[ 81 338]
 ...
 [ 81 379]] MCE CES
[[ 81 383]
 ...
 [ 80 427]] Malnad College of Engineering
[[1271  391]
 ..

ZeroDivisionError: division by zero

In [170]:
len(token_word)
# token_word

175

In [171]:
len(token_poly)
# token_poly

175

In [172]:
image = Image.open(image_path).convert("RGB")
width, height = image.size
print(width,height)

2232 3300


In [173]:
boxes=[]
def normalize(token_ploy, width, height):
    for box in token_ploy:
        boxes.append([
            int(1000 * box[0] / width),
            int(1000 * box[1] / height),
            int(1000 * box[2] / width),
            int(1000 * box[3] / height),
            ])
normalize(token_poly,width,height)

In [174]:
pprint.pprint(boxes)

[[36, 4, 169, 20],
 [181, 4, 302, 20],
 [36, 4, 169, 20],
 [181, 4, 302, 20],
 [566, 2, 620, 15],
 [629, 2, 652, 15],
 [685, 1, 718, 16],
 [724, 1, 751, 16],
 [758, 1, 778, 16],
 [784, 1, 834, 13],
 [36, 19, 72, 34],
 [81, 19, 160, 34],
 [169, 19, 223, 34],
 [232, 19, 293, 34],
 [302, 19, 374, 34],
 [383, 19, 436, 34],
 [566, 17, 669, 31],
 [784, 16, 878, 31],
 [568, 32, 630, 44],
 [638, 32, 670, 44],
 [786, 31, 873, 43],
 [882, 31, 899, 43],
 [908, 31, 969, 43],
 [38, 34, 93, 45],
 [103, 34, 112, 45],
 [121, 34, 176, 45],
 [36, 46, 64, 60],
 [74, 46, 84, 60],
 [93, 46, 198, 60],
 [207, 46, 303, 60],
 [37, 60, 140, 73],
 [150, 60, 305, 73],
 [570, 62, 641, 71],
 [649, 62, 672, 71],
 [680, 62, 688, 71],
 [696, 62, 736, 71],
 [787, 60, 828, 70],
 [836, 60, 918, 70],
 [34, 72, 79, 89],
 [88, 72, 124, 89],
 [133, 72, 142, 89],
 [151, 72, 232, 89],
 [241, 72, 277, 89],
 [286, 72, 295, 89],
 [304, 72, 322, 89],
 [35, 86, 69, 101],
 [76, 86, 110, 101],
 [117, 86, 138, 101],
 [569, 90, 623, 10

In [175]:
image = Image.open(image_path).convert("RGB")

In [176]:
encoding = processor(
    images=image,
    text=token_word,
    boxes=boxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True
)

In [177]:
# Move tensors to the same device as model
for k, v in encoding.items():
    encoding[k] = v.to(device)

In [179]:
# Get predictions
outputs = model(**encoding)
print(outputs)
# The model outputs logits for each token
logits = outputs.logits  # shape: [batch_size, seq_len, num_labels]
print(logits)
# Get predicted class for each token
predictions = torch.argmax(logits, dim=-1)
print(predictions)

TokenClassifierOutput(loss=None, logits=tensor([[[ 0.2679,  0.1866],
         [ 0.1295, -0.0885],
         [ 0.1233, -0.3664],
         ...,
         [ 0.0361,  0.2008],
         [ 0.0361,  0.2008],
         [ 0.0361,  0.2008]]], device='cuda:0', grad_fn=<ViewBackward0>), hidden_states=None, attentions=None)
tensor([[[ 0.2679,  0.1866],
         [ 0.1295, -0.0885],
         [ 0.1233, -0.3664],
         ...,
         [ 0.0361,  0.2008],
         [ 0.0361,  0.2008],
         [ 0.0361,  0.2008]]], device='cuda:0', grad_fn=<ViewBackward0>)
tensor([[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
         0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0,
         0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0,
         0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,

In [183]:
# Suppose your model has these label mappings
labels = model.config.id2label

predicted_labels = [labels[p.item()] for p in predictions[0][:len(token_word)]]

for token_word, label in zip(token_word, predicted_labels):
    print(token_word, "->", label)

PARMESHWARA -> LABEL_0
STATIONERY -> LABEL_0
PARMESHWARA -> LABEL_0
STATIONERY -> LABEL_0
Involce -> LABEL_0
No. -> LABEL_1
e-Way -> LABEL_0
Bill -> LABEL_0
No. -> LABEL_0
Dated -> LABEL_0
Near -> LABEL_0
Hasanamba -> LABEL_0
Temple -> LABEL_0
Circle, -> LABEL_1
Balegara -> LABEL_0
Street -> LABEL_0
PS/25-26/1712 -> LABEL_0
18-Sep-26 -> LABEL_0
Delivery -> LABEL_0
Note -> LABEL_0
Mode/Terms -> LABEL_0
of -> LABEL_0
Payment -> LABEL_1
Hassan -> LABEL_0
- -> LABEL_0
573201 -> LABEL_0
Mob -> LABEL_0
: -> LABEL_0
9448919881, -> LABEL_0
7760638884 -> LABEL_0
GSTIN/UIN: -> LABEL_1
29AFHPM3915J1Z5 -> LABEL_0
Reference -> LABEL_1
No. -> LABEL_0
& -> LABEL_0
Date. -> LABEL_0
Other -> LABEL_0
References -> LABEL_0
State -> LABEL_0
Name -> LABEL_0
: -> LABEL_0
Kamataka, -> LABEL_0
Code -> LABEL_0
: -> LABEL_0
29 -> LABEL_0
Buyer -> LABEL_0
(Bill -> LABEL_0
to) -> LABEL_0
Buyer's -> LABEL_0
Order -> LABEL_0
No. -> LABEL_0
Dated -> LABEL_0
MCE -> LABEL_0
CES -> LABEL_0
Malnad -> LABEL_0
College -> 